# Transformer Basic PEFT: AdaLoRA (TensorFlow)
**Date**: 2026-05-31  
**Objective**: Basic transformer-based PEFT using AdaLoRA pattern in TensorFlow.

## Common Logic Highlight
- Same runtime switch policy using USE_GPU from configs/runtime.env.
- Same project dataset and metrics for cross-framework comparison.
- Type focus: rank-adaptive low-rank adaptation.

In [ ]:
import os
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
def load_runtime_env() -> None:
    candidate_paths = [
        os.path.join(os.getcwd(), 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), 'configs', 'runtime.env.example'),
        os.path.join(os.getcwd(), '..', 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), '..', 'configs', 'runtime.env.example'),
    ]
    env_loaded = False
    for path in candidate_paths:
        if not os.path.exists(path):
            continue
        with open(path, 'r', encoding='utf-8') as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, value = line.split('=', 1)
                os.environ[key.strip()] = value.strip().strip("\"'")
        env_loaded = True
        break
    if not env_loaded and 'USE_GPU' not in os.environ:
        os.environ['USE_GPU'] = '1'

def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {'0', 'false', 'no', 'off'}

load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv('USE_GPU', '1'))
GPU_AVAILABLE = len(tf.config.list_physical_devices('GPU')) > 0
RUNTIME_DEVICE = 'gpu' if USE_GPU and GPU_AVAILABLE else 'cpu'
print(f'USE_GPU={int(USE_GPU)} | runtime_device={RUNTIME_DEVICE}')

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    vocab_size: int = 1024
    seq_len: int = 24
    emb_dim: int = 64
    heads: int = 4
    ff_dim: int = 128
    adapter_rank: int = 8
    prompt_len: int = 2
    num_labels: int = 3
    batch_size: int = 4
    epochs: int = 3
    lr: float = 1e-3
    train_size: float = 0.8

cfg = ExperimentConfig()

## Before vs After PEFT (Code Example)
- Before PEFT: full fine-tuning (all backbone parameters trainable).
- After PEFT: freeze backbone and train only prompt/adapter/head.

In [ ]:
class DemoFullFineTune(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.emb = tf.keras.layers.Embedding(1024, 64)
        self.ff = tf.keras.layers.Dense(64, activation='relu')
        self.cls = tf.keras.layers.Dense(3)
    def call(self, x):
        h = self.emb(x)
        h = tf.reduce_mean(h, axis=1)
        return self.cls(self.ff(h))

class DemoPeft(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.emb = tf.keras.layers.Embedding(1024, 64)
        self.ff = tf.keras.layers.Dense(64, activation='relu')
        self.adapter_down = tf.keras.layers.Dense(8, use_bias=False)
        self.adapter_up = tf.keras.layers.Dense(64, use_bias=False)
        self.cls = tf.keras.layers.Dense(3)
    def call(self, x):
        h = self.emb(x)
        h = tf.reduce_mean(h, axis=1)
        base = self.ff(h)
        adapted = base + self.adapter_up(self.adapter_down(base))
        return self.cls(adapted)

def count_trainable_tf(model: tf.keras.Model) -> int:
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

sample_ids = tf.ones((2, 12), dtype=tf.int32)
before_impl = DemoFullFineTune(); _ = before_impl(sample_ids)
after_impl = DemoPeft(); _ = after_impl(sample_ids)
after_impl.emb.trainable = False; after_impl.ff.trainable = False
before_params = count_trainable_tf(before_impl)
after_params = count_trainable_tf(after_impl)
reduction = 1 - (after_params / before_params)
print({'before_trainable': before_params, 'after_trainable': after_params, 'reduction_ratio': round(float(reduction), 4)})

In [ ]:
samples = [
    ('Server not reachable after deployment', 2),
    ('Password reset email not received', 1),
    ('Dashboard typo in heading', 0),
    ('Payment API timing out for premium users', 2),
    ('Need help changing profile picture', 0),
    ('CPU usage spikes to 100 percent hourly', 2),
    ('Can we export reports to CSV?', 0),
    ('Intermittent login failures for SSO users', 2),
    ('Dark mode icon is slightly misaligned', 0),
    ('Data sync lag observed in EU region', 1),
    ('Mobile app crashes on checkout page', 2),
    ('Feature request: bulk archive tickets', 0),
    ('Webhook retries causing duplicate events', 1),
    ('Fraud alert queue delayed by 5 minutes', 2),
    ('Question about invoice date format', 0),
    ('Latency increased after model update', 1),
]
df = pd.DataFrame(samples, columns=['text', 'label'])
df['label_name'] = df['label'].map({0: 'low', 1: 'medium', 2: 'high'})
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split_idx = int(len(df) * cfg.train_size)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
display(train_df.head(3))

In [ ]:
def build_vocab(texts: list[str], vocab_size: int) -> dict[str, int]:
    vocab = {'<pad>': 0, '<unk>': 1}
    for text in texts:
        for token in text.lower().split():
            if token not in vocab and len(vocab) < vocab_size:
                vocab[token] = len(vocab)
    return vocab

def encode_text(text: str, vocab: dict[str, int], seq_len: int) -> list[int]:
    ids = [vocab.get(t, vocab['<unk>']) for t in text.lower().split()]
    ids = ids[:seq_len]
    ids += [vocab['<pad>']] * max(0, seq_len - len(ids))
    return ids

vocab = build_vocab(train_df['text'].tolist(), cfg.vocab_size)
x_train = np.array([encode_text(t, vocab, cfg.seq_len) for t in train_df['text']], dtype=np.int32)
x_test = np.array([encode_text(t, vocab, cfg.seq_len) for t in test_df['text']], dtype=np.int32)
y_train = train_df['label'].to_numpy(dtype=np.int32)
y_test = test_df['label'].to_numpy(dtype=np.int32)

In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, emb_dim: int, heads: int, ff_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.attn = tf.keras.layers.MultiHeadAttention(num_heads=heads, key_dim=emb_dim // heads)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='relu'),
            tf.keras.layers.Dense(emb_dim),
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
    def call(self, x, training=False):
        h = self.attn(x, x, training=training)
        x = self.norm1(x + h)
        h2 = self.ffn(x, training=training)
        return self.norm2(x + h2)

class SoftPromptConcat(tf.keras.layers.Layer):
    def __init__(self, prompt_len: int, emb_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.prompt_len = prompt_len
        self.emb_dim = emb_dim
    def build(self, input_shape):
        self.soft_prompt = self.add_weight(
            name='soft_prompt',
            shape=(self.prompt_len, self.emb_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )
    def call(self, token_embeddings):
        batch_size = tf.shape(token_embeddings)[0]
        prompt = tf.expand_dims(self.soft_prompt, axis=0)
        prompt = tf.repeat(prompt, repeats=batch_size, axis=0)
        return tf.concat([prompt, token_embeddings], axis=1)

class AdapterLayer(tf.keras.layers.Layer):
    def __init__(self, hidden_dim: int, rank: int, **kwargs):
        super().__init__(**kwargs)
        self.down = tf.keras.layers.Dense(rank, use_bias=False)
        self.up = tf.keras.layers.Dense(hidden_dim, use_bias=False)
    def call(self, x):
        return x + self.up(self.down(x))

inp = tf.keras.Input(shape=(cfg.seq_len,), dtype=tf.int32)
tok = tf.keras.layers.Embedding(cfg.vocab_size, cfg.emb_dim, name='tok_emb')(inp)
x = SoftPromptConcat(cfg.prompt_len, cfg.emb_dim, name='soft_prompt_concat')(tok)
x = TransformerBlock(cfg.emb_dim, cfg.heads, cfg.ff_dim, name='encoder')(x)
pooled = tf.keras.layers.GlobalAveragePooling1D(name='pool')(x)
adapted = AdapterLayer(cfg.emb_dim, cfg.adapter_rank, name='adapter')(pooled)
out = tf.keras.layers.Dense(cfg.num_labels, activation='softmax', name='classifier')(adapted)
model = tf.keras.Model(inp, out)
for layer in model.layers:
    if layer.name in {'soft_prompt_concat', 'adapter', 'classifier'}:
        layer.trainable = True
    else:
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(cfg.lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
pred = model.predict(x_test, verbose=0).argmax(axis=1)
metrics = {
    'accuracy': float(accuracy_score(y_test, pred)),
    'macro_f1': float(f1_score(y_test, pred, average='macro')),
}
print(metrics)

In [ ]:
metric_names = ['accuracy', 'macro_f1']
metric_values = [metrics[k] for k in metric_names]
plt.figure(figsize=(6, 4))
plt.bar(metric_names, metric_values)
plt.ylim(0.0, 1.0)
plt.title('Transformer Basic AdaLoRA TensorFlow - Metrics')
plt.ylabel('Score')
plt.show()

## Summary
- AdaLoRA perspective: rank-adaptive low-rank adaptation.
- Transformer backbone remains mostly frozen for PEFT behavior.
- Shared dataset and metrics keep comparison aligned across frameworks.